# **Thực nghiệm 5% - extract → TRAIN TOÀN BỘ MA TRẬN (1 file, CHẠY 2 LẦN)**

Đây là notebook cho **mức báo cáo chính của repo: 5%** (train 5% split train, **dev/test LUÔN
full**). Lần 2 chạy TOÀN BỘ ma trận thí nghiệm ở 5% bằng `run_all.py` (6 encoder × XE+SCST + 8
thuật toán/biến thể RL + reward ablation + latency + baseline sàn).

**Cách dùng:** đổi `MODE` ở Cell tiếp theo và ĐỔI ACCELERATOR giữa 2 lần chạy.

| Lần | `MODE` | Accelerator | Add Input | Internet |
|---|---|---|---|---|
| 1 – extract | `"extract"` | **None (CPU)** | dataset **ẢNH** (`rwth-phoenix-2014-t`) | ON |
| 2 – train | `"train"` | **GPU T4** | dataset **POSE** bạn tạo từ `poses_5pct.zip` | ON |

**Quy trình:** chạy lần 1 (CPU) → tab **Output** tải `poses_5pct.zip` → **New Dataset** (upload zip,
Kaggle tự giải nén) → đổi `MODE="train"`, đổi Accelerator sang GPU, **Add** dataset pose đó → chạy
lần 2. `run_all.py` là RESUMABLE: hết giờ session cứ chạy lại đúng cell, các bước xong tự bỏ qua.

**Link dataset**
- Image: `https://www.kaggle.com/datasets/thesisacc2/rwth-phoenix-2014-t`
- Annotaion: `https://www.kaggle.com/datasets/thesisacc2/phoenix-2014t-annotations`

In [ ]:
# ================== CÔNG TẮC ==================
MODE = "extract"   # <-- lần 1: "extract"  |  lần 2: đổi thành "train"
# ============================================

import sys, subprocess
def _pip(*a): subprocess.run([sys.executable, "-m", "pip", *a], check=True)

if MODE == "extract":
    # gỡ sạch tf/keras/mediapipe cũ để mediapipe 0.10.14 (có solutions.holistic) cài sạch
    _pip("uninstall", "-y", "-q", "tensorflow", "tensorflow-cpu", "tf-keras", "keras",
         "mediapipe", "mediapipe-nightly")
    _pip("install", "-q", "mediapipe==0.10.14", "opencv-python-headless", "tqdm")
else:  # train
    # sentencepiece + sacrebleu = bắt buộc; bert-score chỉ cần nếu bật Reward BERTScore (weight>0)
    _pip("install", "-q", "sentencepiece", "sacrebleu", "bert-score")
print("MODE =", MODE, "| cài deps xong")

In [ ]:
# Lấy code mới nhất từ GitHub (cả 2 lần đều cần)
import os, subprocess
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "Sign-Language-REL_code"])
subprocess.run(["git", "clone", "https://github.com/linhxm/Sign-Language-REL_code.git"], check=True)
os.chdir("/kaggle/working/Sign-Language-REL_code")
print("cwd:", os.getcwd())

In [ ]:
# ---------- LẦN 1: EXTRACT (Accelerator = None / CPU) ----------
if MODE == "extract":
    import os, sys, subprocess, shutil
    # Thư mục dataset ẢNH: nơi chứa TRỰC TIẾP dev/ train/ test/
    IMG_DIR = "/kaggle/input/datasets/thesisacc2/rwth-phoenix-2014-t"

    # 1) Danh sách tên cần: 5% train (đúng seed loader) + full dev + full test
    subprocess.run([sys.executable, "scripts/make_subset_names.py",
                    "--subset", "0.05", "--out", "/kaggle/working/subset_names.txt"], check=True)

    # 2) Extract CHỈ các sequence đó (không phải cả 8257)
    subprocess.run([sys.executable, "data/extract_poses.py",
                    "--input_dir", IMG_DIR,
                    "--out_dir", "/kaggle/working/poses",
                    "--names_file", "/kaggle/working/subset_names.txt",
                    "--workers", "0"], check=True)

    # 3) Đóng gói thành 1 file .zip để TẢI VỀ từ tab Output
    n = len([f for f in os.listdir("/kaggle/working/poses") if f.endswith(".npz")])
    shutil.make_archive("/kaggle/working/poses_5pct", "zip", "/kaggle/working/poses")
    print(f"\n✅ Extract xong: {n} file .npz")
    print("→ /kaggle/working/poses_5pct.zip : vào tab OUTPUT tải về, tạo New Dataset (upload zip),")
    print("  rồi chạy lại notebook với MODE='train' + Accelerator GPU + Add dataset pose đó.")
else:
    print("bỏ qua extract (MODE != 'extract')")

In [ ]:
# ---------- LẦN 2: TRAIN (Accelerator = GPU T4, Add Input = dataset POSE) ----------
if MODE == "train":
    import os, sys, glob, subprocess
    # Tự tìm thư mục pose đã mount (dataset bạn up từ poses_5pct.zip)
    npz = glob.glob("/kaggle/input/**/*.npz", recursive=True)
    assert npz, "Chưa thấy .npz nào trong /kaggle/input — đã Add dataset pose (từ poses_5pct.zip) chưa?"
    POSE_DIR = os.path.dirname(npz[0])
    print(f"pose_cache_dir = {POSE_DIR}  ({len(npz)} file .npz)")

    # Trỏ config sang thư mục pose vừa mount
    s = open("configs/config.py", encoding="utf-8").read()
    s = s.replace('pose_cache_dir: str = "/kaggle/input/phoenix-poses"',
                  f'pose_cache_dir: str = "{POSE_DIR}"')
    open("configs/config.py", "w", encoding="utf-8").write(s)

    # TOÀN BỘ ma trận ở 5% = 6 encoder + 8 RL + reward ablation + latency + baseline sàn (~6-9h).
    # RESUMABLE: hết giờ session cứ chạy lại cell này, bước đã xong tự bỏ qua (marker .done_*).
    # Muốn TEST NHANH luồng trước (chỉ 1 XE + 1 SCST, <1h): đổi GROUPS = "core".
    GROUPS = "all"
    subprocess.run([sys.executable, "run_all.py",
                    "--subset", "0.05", "--groups", GROUPS], check=True)
else:
    print("bỏ qua train (MODE != 'train')")

In [ ]:
# ---------- Xem kết quả: bảng render ĐẸP (pandas) + biểu đồ hiển thị INLINE ----------
if MODE == "train":
    import os, glob
    import pandas as pd
    from IPython.display import display, Image, Markdown

    WORK = "/kaggle/working"
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.max_colwidth", 40)

    def _show_csv(path, title, sort=None):
        if not os.path.exists(path):
            return
        df = pd.read_csv(path)
        if sort:
            df = df.sort_values([c for c in sort if c in df.columns], na_position="last")
        display(Markdown(f"### {title}"))
        display(df.reset_index(drop=True))   # DataFrame -> Kaggle render thành bảng HTML đẹp

    # 1) Bảng GỘP mọi run (thay cho việc print markdown thô -> giờ là bảng thật)
    _show_csv(os.path.join(WORK, "comparison_table.csv"),
              "Bảng gộp — tất cả run", sort=["subset_pct", "encoder", "method"])

    # 2) 6 bảng báo cáo đã lọc sẵn theo từng câu hỏi so sánh
    TB = os.path.join(WORK, "report", "tables")
    for name, title in [
        ("table_main",      "Exp 1/7 — XE vs mọi thuật toán RL (Transformer)"),
        ("table_encoders",  "Exp 4 — so sánh 6 encoder"),
        ("table_reward",    "Exp 9 — reward ablation (4 combo)"),
        ("table_ablations", "REINFORCE / A2C / Curriculum"),
        ("table_baseline",  "Baseline sàn (empty / most-frequent)"),
        ("table_latency",   "Exp 15 — latency / memory / params"),
    ]:
        _show_csv(os.path.join(TB, name + ".csv"), title)

    # 3) Các biểu đồ (PNG) hiển thị NGAY trong notebook
    FG = os.path.join(WORK, "report", "figures")
    pngs = sorted(glob.glob(os.path.join(FG, "*.png")))
    if pngs:
        display(Markdown("### Biểu đồ"))
        for p in pngs:
            display(Markdown(f"**{os.path.basename(p)}**"))
            display(Image(filename=p))
    else:
        print("Chưa có biểu đồ trong", FG, "— chạy xong run_all.py (nó tự gọi make_report) rồi mở lại.")
else:
    print("bỏ qua xem kết quả (MODE != 'train')")